# E1 · Detección Hα

**Spec:** [`docs/spec_E1_codex_halpha_detection.md`](../docs/spec_E1_codex_halpha_detection.md)  |  **Bloque:** E · Resultado  |  **Run por defecto:** `ROXs12b_realigned`

Test de detección de emisión Hα del compañero con matched filter y controles.

| | |
|---|---|
| **Entrada** | `spec_final_object.fits` + controles |
| **Salida (QC/productos)** | `stages/stage_h01_qc.json` |
| **Consume aguas abajo** | E2, E3, G2 (V3) |


## Qué hace E1 y el resultado

E1 busca **emisión de Hα** del compañero con un **matched filter** (plantilla de la línea esperada) y calibra la significancia con **controles** (FAP empírico). Es el **endpoint científico**.

Por método: busca en ±500 km/s alrededor de Hα (6562.8 Å, rv_sys −7) → un `z` del matched filter. La **FAP** = fracción de los 33 máximos nulos (posiciones de control) que superan el pico del objeto. **Criterio de detección:** `global_fap < 0.01` **Y** un par admisible (psffit+aperture) **Y** rv dentro de la LSF.

**VEREDICTO = `non_detection`** (`no_method_passes_global_fap`): ningún método pasa. Los picos del objeto (z 1.0–3.4) caen **dentro de sus distribuciones nulas** (FAP 0.62–0.97 ≫ 0.01). El pico de **psffit z=3.41** parece 'algo', pero sus nulos llegan a 8 (borde ruidoso, ~10× ruido) → FAP 0.88; además `rv_consistent=False` (v=+128 vs esperado ~−7) y FWHM 10 Å (demasiado ancho para Hα) → **ruido, no línea**.

**Inputs:** LSF 2.383 Å (medida en A4/M2), rv_sys −7 (estimación de literatura), 33 controles → `min_resolvable_fap` ≈ 0.029 (aún >0.01; un FAP<1% estricto necesitaría ~99 controles — salvedad).

**Resultado robusto: no hay señal de acreción en Hα de ROXs 12 B.**


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # o ROXs12b_B_adp para comparar
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_h01_detect.sh --run-id $RUN
```

Ligero–moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id(None)
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_h01_detect.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc('stages/stage_h01_qc.json', RUN_ID)
nb.show(qc, keys=['verdict', 'reason', 'global_fap_lt', 'significant_methods', 'rv_consistent'], title='E1')


## Resultados que llevaron a la conclusión

Veredicto, criterio y el pico/FAP/rv de cada método del `stage_h01_qc.json` + la tabla.


In [ ]:
import pandas as pd
q = nb.load_qc('stages/stage_h01_qc.json', RUN_ID)
v = q['verdict']; cr = q['criterion']
print('VEREDICTO:', v['verdict'], '|', v['reason'], '| métodos significativos:', v['significant_methods'])
print(f"criterio: global_fap<{cr['global_fap_lt']}, par admisible {cr['admissible_pairs']}, rv dentro de LSF")
print(f"línea: Hα {q['line']['rest_A']} Å, rv_sys {q['line']['rv_sys_kms']} km/s, búsqueda ±{q['line']['search_half_width_kms']:.0f} km/s")
print()
d = pd.read_csv(nb.run_dir(RUN_ID) / 'tables' / 'halpha_detection_by_method.csv')
for _, r in d.iterrows():
    print(f"  {r['method']:15s} z={r['matched_z']:.2f}  FAP={r['global_empirical_fap']:.2f}  "
          f"v={r['peak_velocity_kms']:+.0f} km/s  rv_ok={r['rv_consistent']}  fwhm={r['fwhm_A']:.1f} Å")
print(f"\nmin_resolvable_fap = {d['minimum_resolvable_fap'].iloc[0]:.3f} (33 controles; <0.01 necesita ~99)")


## Plot 1 — la no-detección: pico del objeto vs distribución nula

Por método, los 33 **máximos nulos** (matched filter en posiciones de control, gris) y el **pico del objeto** (estrella). En todos, el pico del objeto queda **dentro de la nube nula** → FAP ≫ 0.01. psffit tiene z alto (3.41) pero sus nulos llegan a 8 (borde ruidoso) → FAP 0.88; rv inconsistente.


In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    rd = nb.run_dir(RUN_ID)
    z = np.load(rd / 'stages' / 'stage_h01_null_maxima.npz')
    d = pd.read_csv(rd / 'tables' / 'halpha_detection_by_method.csv').set_index('method')
    methods = ['aperture', 'optimal_psfsub', 'psffit', 'optimal_ls']
    rng = np.random.default_rng(1)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, m in enumerate(methods):
        nulls = z[f'{m}_null_maxima']; x = i + rng.uniform(-0.12, 0.12, nulls.size)
        ax.scatter(x, nulls, s=14, color='0.6', alpha=0.7, label='máximos nulos (33 controles)' if i == 0 else None)
        obj = d.loc[m, 'matched_z']; fap = d.loc[m, 'global_empirical_fap']; rv = d.loc[m, 'rv_consistent']
        ax.scatter(i, obj, s=170, marker='*', color='tab:orange' if rv else 'tab:red', zorder=5,
                   edgecolor='k', label='pico del objeto' if i == 0 else None)
        ax.text(i, obj + 0.35, f'z={obj:.2f}\nFAP={fap:.2f}\nrv_ok={rv}', ha='center', fontsize=7)
    ax.set_xticks(range(len(methods))); ax.set_xticklabels(methods, fontsize=9)
    ax.set_ylabel('z del matched filter (máximo en la ventana Hα)')
    ax.set_title('E1 · no-detección: el pico del objeto queda dentro de la nube nula (FAP >> 0.01)')
    ax.legend(fontsize=8, loc='upper left'); fig.tight_layout()
    outdir = rd / 'plots' / 'e1_halpha'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'detection.png', dpi=110); print('figura ->', outdir / 'detection.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — la región de Hα en el espectro canónico

El espectro psffit (`spec_final_object.fits`) alrededor de Hα con la banda ±1σ empírica y la posición esperada de Hα (6562.8 Å a rv=−7). **No hay línea** por encima del ruido en la posición esperada.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID); wave = 4749.533203125 + 1.25 * np.arange(3681)
    h = fits.open(rd / 'stages' / 'spec_final_object.fits'); flux = np.asarray(h[1].data['flux'], float); h.close()
    sig = np.nanstd(np.load(rd / 'stages' / 'spec_calibrated_psffit_controls.npz')['control_spectra'], axis=0)
    ha = 6562.8 * (1 + (-7.0) / 299792.458)
    w = (wave >= 6400) & (wave <= 6750)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.fill_between(wave[w], -sig[w], sig[w], color='0.85', label='±1σ empírico')
    ax.plot(wave[w], flux[w], lw=0.9, color='tab:blue', label='flujo psffit')
    ax.axvline(ha, color='tab:red', ls=':', label=f'Hα esperado ({ha:.1f} Å)')
    ax.axhline(0, color='0.6', lw=0.6)
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('flujo'); ax.legend(fontsize=8)
    ax.set_title('E1 · región de Hα: sin línea sobre el ruido en la posición esperada')
    fig.tight_layout()
    outdir = rd / 'plots' / 'e1_halpha'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'halpha_region.png', dpi=110); print('figura ->', outdir / 'halpha_region.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **VEREDICTO = `non_detection`** — ningún método supera el FAP global (0.62–0.97 ≫ 0.01); los picos caen dentro de la nube nula. **ENDPOINT CIENTÍFICO: no hay señal de acreción en Hα.**
- El pico psffit (z=3.41) es RV-inconsistente (v=+128) y demasiado ancho (10 Å) → ruido, no línea; su nube nula llega a z=8 (borde ~10× ruido).
- LSF = 2.383 Å (medida, A4/M2), 33 controles → min_resolvable_fap ≈ 0.029 (un FAP<1% estricto necesitaría ~99 controles).


## Conclusión (registrada)

**E1: veredicto `non_detection` — no hay señal de acreción en Hα de ROXs 12 B.**

- **Fecha:** cadena D1 v2 realineado (2026-07-09), con LSF medida.
- **Todos los métodos:** FAP 0.62–0.97 ≫ 0.01; picos del objeto dentro de sus nubes nulas.
- **psffit z=3.41** (el mayor) es rv-inconsistente (+128 km/s) y demasiado ancho (10 Å) → ruido.
- **Inputs:** LSF 2.383 Å medida, 33 controles (min_fap 0.029; ~99 para 1% estricto), rv_sys −7 (literatura).
- **Downstream:** alimenta E3 (límite superior de Ṁ) y G2. Es el endpoint científico del proyecto.
